In [15]:
import pandas as pd
from alpaca_trade_api.rest import REST, TimeFrame, TimeFrameUnit
from datetime import datetime,timedelta, timezone

In [65]:
# initialize alpaca
api = REST(key_id = 'PKU9G2XRSNL8A1FF34C7', secret_key = 'AQe6RsvUdmZykRZCRYsNF5kEogta6DIyCApkoMAl'
          ,base_url = "https://paper-api.alpaca.markets")

In [26]:
trading_key = pd.read_csv('data/trading_key_test_for_prices.csv')

In [32]:
# hardcoding to America/New York for now as convertion to time zone
# returns a timezone'd datetime, so if comparing to a regular date time we will need to remove
# time zone info
def utc_to_local(utc_dt):
    return utc_dt.replace(tzinfo=timezone.utc).astimezone(tz='America/New_York')


def calc_avg_entry_by_ticker(df):
    df['date_time'] = pd.to_datetime(df['date_time'])
    symbol = df['symbol']
    day = df['date_time'].strftime('%Y-%m-%d')
    bars_df = api.get_bars(symbol, TimeFrame(1, TimeFrameUnit.Minute), day, day, adjustment='raw').df.reset_index()
    #bars_df['timestamp_lcl'] = bars_df.apply(lambda x : utc_to_local(x['timestamp']), axis = 1)
    print(bars_df)
    bars_df['timestamp_no_tz'] = bars_df['timestamp'].apply(lambda x : x.replace(tzinfo =None))
    bars_df['max_time'] = df['date_time'] + timedelta(minutes=5)
    bars_df['min_time'] = df['date_time']
    bars_df_date_mask = (bars_df['timestamp_no_tz'] >= bars_df['min_time']) & (bars_df['timestamp_no_tz'] <= bars_df['max_time'])
    print(bars_df)
    bars_df.to_csv('barstest.csv')
    bars_df = bars_df[bars_df_date_mask]
    print(bars_df)
    return bars_df['close'].mean()

In [59]:
trading_key['entry_price'] = trading_key.apply(calc_avg_entry_by_ticker, axis = 1)

Empty DataFrame
Columns: [index]
Index: []


KeyError: 'timestamp'

In [29]:
trading_key

,eth_cust_address,address,chain,symbol,target_position_value,date_time
0,0x0192523cd5beb497121a82a956c09730ed83e11d,0x0192523cd5bEb497121A82a956C09730ed83E11d,SKL,NAT,0.03,4/16/2022 17:12


In [163]:
day = "2022-04-14"
api.get_bars("NAT", TimeFrame(1, TimeFrameUnit.Minute), "2022-04-14", "2022-04-14", adjustment='raw').df.reset_index()


,timestamp,open,high,low,close,volume,trade_count,vwap
0,2022-04-14 08:04:00+00:00,2.85,2.85,2.85,2.85,442,2,2.85000
1,2022-04-14 08:05:00+00:00,2.85,2.85,2.85,2.85,843,7,2.85000
2,2022-04-14 08:10:00+00:00,2.86,2.89,2.86,2.89,200,2,2.87500
3,2022-04-14 08:31:00+00:00,2.91,2.91,2.91,2.91,500,1,2.91000
4,2022-04-14 08:38:00+00:00,2.87,2.87,2.87,2.87,100,1,2.87000
...,...,...,...,...,...,...,...,...
438,2022-04-14 20:55:00+00:00,2.75,2.75,2.75,2.75,500,1,2.75000
439,2022-04-14 21:45:00+00:00,2.70,2.70,2.70,2.70,6808,1,2.70000
440,2022-04-14 22:39:00+00:00,2.76,2.76,2.76,2.76,150,1,2.76000
441,2022-04-14 23:09:00+00:00,2.78,2.80,2.78,2.80,1424,6,2.79434


In [70]:
from datetime import date as todaysDate
from datetime import datetime as todaysDateTime
Today_date  =  todaysDate.today()
Today_time  =  todaysDateTime.min.time()
Today_datetime = todaysDateTime.combine(Today_date, Today_time)
print(Today_datetime)

2022-06-16 00:00:00


In [160]:

def get_delayed_trade_calendar(trade_key):
    # get min trade key date
    start_date = pd.to_datetime(trade_key['date_time']).dt.date.min()

    # alpaca calendar to denote business days
    delayed_trade_date_time_df = pd.DataFrame({'delayed_trade_date_time':
                                                   [datetime.combine(x.date, x.open).astimezone(pytz.utc).replace(
                                                       tzinfo=None)
                                                    for x in api.get_calendar("2022-04-11")]})

    # agnostic date range of all days between min trade key date and max alpaca business date
    date_df = pd.DataFrame({'date': pd.date_range(start="2022-04-11",
                                                  end=delayed_trade_date_time_df['delayed_trade_date_time'].max())})

    # conversion for merging
    delayed_trade_date_time_df['delayed_trade_date'] = \
        pd.to_datetime(delayed_trade_date_time_df['delayed_trade_date_time']).dt.date

    # conversion for merging
    delayed_trade_date_time_df['delayed_trade_date'] = pd.to_datetime(
        delayed_trade_date_time_df['delayed_trade_date'])

    # merging agnostic dates with business dates
    bus_date_key_df = pd.merge(left=date_df, right=delayed_trade_date_time_df,
                               how='left', left_on='date',
                               right_on='delayed_trade_date')

    # backfilling empty merge results so that the next business day propgates backwards for non business days
    bus_date_key_df.fillna(method='bfill', inplace=True)
    bus_date_key_df.dropna(inplace=True)

    return bus_date_key_df


In [161]:
trading_key = pd.read_csv('data/trading_key_test_for_prices.csv')

In [162]:
get_delayed_trade_calendar(trading_key)

,date,delayed_trade_date_time,delayed_trade_date
0,2022-04-11,2022-04-11 13:30:00,2022-04-11
1,2022-04-12,2022-04-12 13:30:00,2022-04-12
2,2022-04-13,2022-04-13 13:30:00,2022-04-13
3,2022-04-14,2022-04-14 13:30:00,2022-04-14
4,2022-04-15,2022-04-18 13:30:00,2022-04-18
...,...,...,...
2810,2029-12-20,2029-12-20 14:30:00,2029-12-20
2811,2029-12-21,2029-12-21 14:30:00,2029-12-21
2812,2029-12-22,2029-12-24 14:30:00,2029-12-24
2813,2029-12-23,2029-12-24 14:30:00,2029-12-24


In [82]:
import pytz
from datetime import datetime

In [91]:
def convert_datetime_timezone(dt, tz1, tz2):
    tz1 = pytz.timezone(tz1)
    tz2 = pytz.timezone(tz2)
    dt = datetime.strptime(dt,"%Y-%m-%d %H:%M:%S")
    dt = tz1.localize(dt)
    dt = dt.astimezone(tz2)
    dt = dt.strftime("%Y-%m-%d %H:%M:%S")
    return dt

In [97]:
open_date = api.get_calendar("2022-04-14")[0].date
open_time = api.get_calendar("2022-04-14")[0].open
open_date_time = datetime.combine(open_date,open_time)
utc_dt = open_date_time.astimezone(pytz.utc)
print(utc_dt)

2022-04-14 13:30:00+00:00


In [159]:
api.get_calendar("2022-04-11")

[Calendar({   'close': '16:00',
     'date': '2022-04-11',
     'open': '09:30',
     'session_close': '2000',
     'session_open': '0400'}),
 Calendar({   'close': '16:00',
     'date': '2022-04-12',
     'open': '09:30',
     'session_close': '2000',
     'session_open': '0400'}),
 Calendar({   'close': '16:00',
     'date': '2022-04-13',
     'open': '09:30',
     'session_close': '2000',
     'session_open': '0400'}),
 Calendar({   'close': '16:00',
     'date': '2022-04-14',
     'open': '09:30',
     'session_close': '2000',
     'session_open': '0400'}),
 Calendar({   'close': '16:00',
     'date': '2022-04-18',
     'open': '09:30',
     'session_close': '2000',
     'session_open': '0400'}),
 Calendar({   'close': '16:00',
     'date': '2022-04-19',
     'open': '09:30',
     'session_close': '2000',
     'session_open': '0400'}),
 Calendar({   'close': '16:00',
     'date': '2022-04-20',
     'open': '09:30',
     'session_close': '2000',
     'session_open': '0400'}),
 Calen

In [165]:
datetime(1970,1,1)

datetime.datetime(1970, 1, 1, 0, 0)

In [166]:
pd.DataFrame({'timestamp': datetime(1970,1,1),'close': 0})

ValueError: If using all scalar values, you must pass an index